In [2]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# 1) Static input–output curves
# ---------------------------
x = np.linspace(-2.5, 2.5, 1000)

# Linear
k = 1.0
y_linear = k * x

# Saturation
def sat(x, limit=1.0):
    return np.clip(x, -limit, limit)
y_sat = sat(k * x, 1.0)

# Dead zone
d = 0.6
def dead_zone(x, d):
    y = np.zeros_like(x)
    y[x > d]  = x[x > d] - d
    y[x < -d] = x[x < -d] + d
    return y
y_dead = k * dead_zone(x, d)

# Hysteresis with backlash (play operator)
b = 0.5
x_sweep_up   = np.linspace(-2.5,  2.5, 600)
x_sweep_down = np.linspace( 2.5, -2.5, 600)
x_loop = np.concatenate([x_sweep_up, x_sweep_down])

def backlash_play_operator(x, b):
    y = np.zeros_like(x)
    y[0] = np.clip(x[0], -b, b)
    for i in range(1, len(x)):
        if x[i] > y[i-1] + b:
            y[i] = x[i] - b
        elif x[i] < y[i-1] - b:
            y[i] = x[i] + b
        else:
            y[i] = y[i-1]
    return y

y_loop = backlash_play_operator(x_loop, b)

plt.figure(figsize=(7,5))
plt.plot(x, y_linear, label="Linear: y=kx")
plt.plot(x, y_sat, label="Saturation")
plt.plot(x, y_dead, label="Dead zone")
plt.plot(x_loop, y_loop, label="Hysteresis (backlash)")
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("Input")
plt.ylabel("Output")
plt.title("Static Input–Output Characteristics: Linear vs Nonlinear")
plt.legend(loc="best")
plt.grid(True, linewidth=0.3)
plt.tight_layout()
plt.savefig("linear_vs_nonlinear_static.png", dpi=200)
plt.close()

# -------------------------------------------
# 2) Dynamic step response: linear vs nonlinear
# -------------------------------------------
T = 10.0
dt = 0.001
t = np.arange(0.0, T, dt)

# Step input
u = np.zeros_like(t)
u[t >= 1.0] = 1.5

# Linear system: dy/dt = -a*y + b*u
a_lin, b_lin = 1.0, 1.0
y_lin = np.zeros_like(t)
for i in range(1, len(t)):
    y_lin[i] = y_lin[i-1] + dt * (-a_lin * y_lin[i-1] + b_lin * u[i-1])

# Nonlinear system: dy/dt = -a*y - c*y^3 + b*sat(u)
a_nl, b_nl, c_nl = 1.0, 1.0, 0.5
def sat_u(u, limit=1.0):
    return np.clip(u, -limit, limit)

y_nl = np.zeros_like(t)
for i in range(1, len(t)):
    u_sat = sat_u(u[i-1], limit=1.0)
    y_nl[i] = y_nl[i-1] + dt * (-a_nl * y_nl[i-1] - c_nl * (y_nl[i-1]**3) + b_nl * u_sat)

plt.figure(figsize=(7,5))
plt.plot(t, y_lin, label="Linear first-order")
plt.plot(t, y_nl, label="Nonlinear: cubic + input saturation")
plt.plot(t, u, linestyle="--", label="Input (step)")
plt.xlabel("Time [s]")
plt.ylabel("Response")
plt.title("Step Response: Linear vs Nonlinear Dynamics")
plt.legend(loc="best")
plt.grid(True, linewidth=0.3)
plt.tight_layout()
plt.savefig("linear_vs_nonlinear_dynamic.png", dpi=200)
plt.close()
